[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lorenzobasile/DeepLearningMHPC/blob/main/1_introduction.ipynb)



# Lab 1: Introduction to MLPs and CNNs

## Before you start: clone the repository

This notebook is part of the course repository and uses files from it (e.g. the figures in the `notebooks_imgs/` folder). **Do not download this notebook on its own**: clone the whole repository first and open the notebook from inside the cloned folder, otherwise the paths to those files will not work.

**Locally** (from a terminal):

```bash
git clone <REPO_URL> mdmc_dl_hands_on
cd mdmc_dl_hands_on
jupyter lab
```

**On Google Colab**, run this in a code cell before anything else:

```python
!git clone <REPO_URL> mdmc_dl_hands_on
%cd mdmc_dl_hands_on
```



Every lab will finish with a small homework that builds on the day's material. We can briefly discuss any doubt or curiosity the following day. The notebooks with the solutions each solution will be published after the whole course has ended.

## How to use this notebook

This is the **student version** of the lab: some code cells are incomplete, and completing them is your job.

- **🛠 Exercise** cells (in markdown) explain what to implement, with hints on which functions to use.
- The code cell right after each exercise contains `# TODO` comments and `...` placeholders: **replace every `...` with your code**. Everything else in the cell is already written for you: read it, you will need it!
- Some exercises are followed by a **✅ Check** cell: run it after your solution. If it prints `All good!` (or the expected values), you can move on.
- Run the cells **in order**: later exercises use variables and functions defined in earlier ones.
- If you get stuck: read the docstrings and comments, the explanation cells above the exercise and the linked documentation, or ask!

## Computational resources

We will not run particularly heavy experiments so we will be running the labs on Google Colab, a service that provides free access GPUs.

For your assignments, it is advisable to switch from Colab to a proper HPC facility.


# Introduction to Colab

Colab is a free service provided by Google for ML research. It is based on Jupyter notebooks that run on a remote server, and it provides free (but limited time) GPU acceleration.

To enable GPU or TPU acceleration just go to `Runtime>Change runtime type` and choose from the menu. Please note that GPU usage is limited in time, so avoid requesting one if you do not really need it.

Inside a code cell you can use `!` to run shell commands, while on your computer you just have to type the command **without** `!`:

```bash
!nvidia-smi    # if you enable GPU acceleration, this command returns information on the GPU
!pip install torch==2.9.0    # just an example, torch is already installed in Colab
!sudo apt-get install gcc    # you can also run sudo commands
!wget <SOMETHING>    # and download data to a temporary memory
!git clone ... # to clone a repository from github
```

## Colab file system

By default, Colab accesses a volatile memory that is erased as soon as your process terminates, but you can interface it with your personal Google Drive to read and write data; this can be useful if you want to premaently store data like models weights or partial results.

```python
from google.colab import drive
drive.mount('/content/drive')
```

```bash
%mkdir drive/MyDrive/DeepLearningMDMC
%cd drive/MyDrive/DeepLearningMDMC
%pwd
```

# Getting started with PyTorch

## What is PyTorch (Refresh) ?

PyTorch (or informally torch) is a Python library specifically built for Deep Learning, that comes with a series of very useful functionalities that make it one of the most popular tools for DL research and application.

Namely, it has many built-in features and modules useful for DL, tensor arithmetic and automatic differentiation features, and it allows for easy GPU acceleration through CUDA.
From [PyTorch documentation](https://docs.pytorch.org/docs/2.15/user_guide/pytorch_main_components.html) we can refresh the main components included.

- **Tensors** - N-dimensional arrays that serve as PyTorch’s fundamental data structure. They support automatic differentiation, hardware acceleration, and provide a comprehensive API for mathematical operations.

- **Autograd** - PyTorch’s automatic differentiation engine that tracks operations performed on tensors and builds a computational graph dynamically to be able to compute gradients.

- **Neural Network API** - A modular framework for building neural networks with pre-defined layers, activation functions, and loss functions. The nn.Module base class provides a clean interface for creating custom network architectures with parameter management.

- **DataLoaders** - Tools for efficient data handling that provide features like batching, shuffling, and parallel data loading. They abstract away the complexities of data preprocessing and iteration, allowing for optimized training loops.



## What we'll do in this first hands on session ?

The objective of this hands on session is:
- how to do deep learning in practise: dataset preparation, cross-validation, hyperparameter optimization.
- train MLPs and CNNs for image classification (supervised learning setting)
- evaluate perfomances

## Library import

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision.transforms import v2
from torchvision.datasets import MNIST
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
from skorch.dataset import ValidSplit

# use the best available device: NVIDIA GPU (cuda) > Apple Silicon GPU (mps) > CPU
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Using device:", DEVICE)

### Device: CPU or GPU

By default PyTorch runs on the CPU. Neural networks are much faster on a GPU, so `DEVICE` is set to the best device available: `cuda` (NVIDIA GPU, e.g. on Colab with `Runtime > Change runtime type > GPU`), `mps` (Apple Silicon) or `cpu`.

The rule is simple: **the model and the data must be on the same device**.
- Model: `model.to(DEVICE)`, before creating the optimizer.
- Data: `x.to(DEVICE)` and `y.to(DEVICE)` for every mini-batch, inside the training and evaluation loops.

Results that go back to NumPy, scikit-learn or matplotlib must be on the CPU first (`.cpu()`).

## Dataset preparation

We'll start to prepare the [MNIST dataset](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.MNIST.html): $28 \times 28$ grayscale images of handwritten digits, 60k for training and 10k for testing. This is a very easy dataset, but with more complex datasets this phase is one of the more boring but important ones, because in DL there is a golden rule: *Garbage in, garbage out*.

In order we'll:

1. **Download** the dataset with `torchvision.datasets.MNIST`.
2. **Convert** the images to tensors with values in $[0, 1]$.
3. **Compute** the mean and standard deviation of the pixels on the training set.
4. **Normalize** the images using these values.
5. **Split** the training set into training and validation sets and create the `DataLoader`s.

### Transforms

A [transform](https://docs.pytorch.org/vision/stable/transforms.html) is a function applied to every sample when we read it from the dataset (`dataset[i]`). The images are stored on disk in their original format, and the transforms convert them **on the fly** into what the model needs. `v2.Compose([...])` chains several transforms: they are applied in order.

The ones we use:

| Transform | What it does | MNIST image after the transform |
|---|---|---|
| `v2.ToImage()` | converts the original image (a PIL image) into a tensor with shape `(channels, height, width)` | `uint8` tensor of shape `(1, 28, 28)`, values in $\{0, \dots, 255\}$ |
| `v2.ToDtype(torch.float32, scale=True)` | converts to floating point numbers; `scale=True` also divides by 255 | `float32`, values in $[0, 1]$ |
| `v2.Normalize(mean, std)` | for every channel computes $x' = \dfrac{x - \text{mean}}{\text{std}}$ | `float32`, mean $\approx 0$, std $\approx 1$ |

**Why normalize?** Neural networks train faster and more stably when the inputs are centred around 0 and have a similar scale: the gradients are better balanced and the default initialization of the weights assumes inputs of this kind.

**Why compute mean and std on the training set only?** The test set must stay unseen until the final evaluation: using it to compute anything used in training would leak information about it. The same mean and std are then applied to the training, validation and test images.

Transforms can also be used for **data augmentation** (e.g. `v2.RandomRotation`, `v2.RandomCrop`), which creates slightly different versions of the training images at every epoch. We don't need it for MNIST.

### 🛠 Exercise 1 — Normalization values

Complete `find_normalization_values`. The function already collects `n_samples` random images into the tensor `f_images`, of shape `(channels, n_samples, height*width)`: for every channel, all the pixel values of all the sampled images.

What is missing is the last line: return the **mean** and the **standard deviation** of the pixels **for every channel**, i.e. two tensors of shape `(channels,)`.

*Hint:* `tensor.mean(dim=...)` and `tensor.std(dim=...)` accept a tuple of dimensions to reduce over. Which dimensions must disappear, and which one must stay?

`plot_example` (in the same cell) is a plotting helper and is already complete.

In [ ]:
def find_normalization_values(trainset: torch.utils.data.Dataset, n_samples: int) -> torch.Tensor:
  """Estimate the mean and the standard deviation of the pixels of `trainset`, for every channel.

  To be fast, it uses `n_samples` random images instead of the whole dataset: a few thousand
  images are enough for a good estimate. Returns two tensors of shape (channels,).
  """
  if n_samples > len(trainset):
    n_samples = len(trainset)
    print("Setting n_samples to len(trainset)")

  channels, height, width = trainset[0][0].size()

  # create a tensor of size (channels, n_samples, height*width)
  f_images = torch.empty(channels, n_samples, height*width)
  # taking a set of random samples from training set
  idxs = torch.randperm(len(trainset))[:n_samples]
  for processed_count, idx in enumerate(idxs):
    f_images[:, processed_count, :] = trainset[idx.item()][0].flatten()
  # TODO: return the mean and the std of every channel (two tensors of shape (channels,))
  return ...

def plot_example(X: torch.Tensor, y: torch.Tensor, n: int = 5):
    """Plot the images in X with their labels in rows of `n` elements,
    each with a colorbar marking that image's min and max value."""
    rows = -(-len(X) // n)  # ceil division
    fig, axes = plt.subplots(rows, n, figsize=(2.4 * n, 2.2 * rows),
                             squeeze=False, layout="constrained")
    for ax in axes.flat:
        ax.set_axis_off()  # also blanks any unused cells in the last row
    for ax, img, label in zip(axes.flat, X, y):
        img = img.detach().cpu().reshape(28, 28).numpy()
        lo, hi = float(img.min()), float(img.max())
        im = ax.imshow(img, vmin=lo, vmax=hi)
        ax.set_title(label.item() if hasattr(label, "item") else label)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_ticks([lo, hi])
        cbar.set_ticklabels([f"{lo:.3g}", f"{hi:.3g}"])
        cbar.ax.tick_params(labelsize=7)

    plt.close(fig)
    return fig

**Steps 1–3: download, convert and compute the normalization values.** We first load the training set with only the first two transforms (values in $[0, 1]$) and use it to compute the mean and standard deviation of the pixels.

### 🛠 Exercise 2 — Transforms and normalization values

1. Build `transforms` with `v2.Compose([...])`, chaining the **first two** transforms of the table above (`v2.ToImage()` and `v2.ToDtype(...)`), so that the images become `float32` tensors with values in $[0, 1]$. Remember `scale=True`!
2. Call `find_normalization_values` on `mnist_train` with `10000` samples to get `mean` and `std`.

Expected result: a mean of about `0.13` and a std of about `0.31` (they change slightly at every run, because the samples are random).

In [ ]:
# TODO: chain the transforms: PIL image -> tensor -> float32 in [0, 1]
transforms = v2.Compose([
    ...
])
mnist_train = MNIST('datasets', train=True, download=True, transform=transforms)

# TODO: estimate mean and std on 10000 random training images
mean, std = ...
mean, std

In [ ]:
# ✅ Check
img = mnist_train[0][0]
assert img.dtype == torch.float32 and img.min() >= 0 and img.max() <= 1, "the images must be float32 with values in [0, 1]"
assert mean.shape == (1,) and std.shape == (1,), "mean and std must have shape (channels,) = (1,): check the dims you reduce over"
assert abs(mean.item() - 0.13) < 0.01 and abs(std.item() - 0.31) < 0.01, "unexpected values for mean and std"
print("All good!")

Let's look at some images: the colorbar next to each image shows its minimum and maximum value, which are now 0 and 1.

In [ ]:
X, y = zip(*(mnist_train[i] for i in range(5)))
plot_example(torch.stack(X), torch.tensor(y))

**Step 4: normalize.** We create the transforms again, adding `v2.Normalize` with the values we just computed, and we load both the training and the test set with them.

### 🛠 Exercise 3 — Normalize

1. Add `v2.Normalize` with the `mean` and `std` you just computed as the **last** transform.
2. Load the test set too (`train=False`) **with the same transforms**: the test images must be preprocessed exactly like the training ones, using the statistics of the training set.

In [ ]:
transforms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    ...  # TODO: normalize with the mean and std computed above
])
mnist_train = MNIST('datasets', train=True, download=True, transform=transforms)
mnist_test = ...  # TODO: the MNIST test set, with the same transforms

In [ ]:
# ✅ Check: after normalization the pixels should have mean ~0 and std ~1
x = torch.stack([mnist_train[i][0] for i in range(2000)])
print(f"mean: {x.mean():.3f} (should be close to 0) - std: {x.std():.3f} (should be close to 1)")
assert abs(x.mean()) < 0.1 and abs(x.std() - 1) < 0.1, "the images are not normalized"
assert len(mnist_test) == 10000, "mnist_test must be the test set (train=False)"
print("All good!")

In [ ]:
X, y = zip(*(mnist_train[i] for i in range(5)))
plot_example(torch.stack(X), torch.tensor(y))

### DataLoader

Now that we have created the dataset let's create the DataLoader from `torch.utils.data.DataLoader`. We'll use the dataloader to split data into batches and shuffle them 
To train any PyTorch model, it is useful to handle data through a `torch.utils.data.DataLoader` object. A `DataLoader` is an iterable wrapped around a `Dataset` object that allows to easily run through your data in batches.

For any specific need, you can build your own `Dataset` class. To make it work properly, you always have to implement three functions: `__init__`, `__len__` and `__getitem__`. More info on this [here](https://pytorch.org/docs/stable/data.html).

Important parameters in the DataLoader are:

- batch_size: int value that tells how many samples per batch to use
- shuffle: boolean value that tells the loader to randomize

NB: during training we want randomness otherwise the models learns how to respond just memorizing the sequence of labels

### 🛠 Exercise 4 — DataLoaders

The split into training and validation sets is given. Create the three `DataLoader`s, all with `batch_size=BATCH_SIZE`:
- `mnist_train_loader` on `mnist_train_split`;
- `mnist_val_loader` on `mnist_val`;
- `mnist_test_loader` on `mnist_test`.

Think about `shuffle`: which loaders should shuffle the data? (Re-read the *NB* above: shuffling matters during **training**. For validation and test the order does not change the result, and a fixed order makes the evaluation reproducible.)

In [ ]:
# split the original training set into training and validation sets
# the validation set is used to monitor the model during training,
# the test set is used only once, at the end, to measure the final performance
# cv=5 -> 5 folds: 1 fold (1/5 = 12k images) is used for validation, the other 4 (48k images) for training
valid_split = ValidSplit(cv=5, stratified=False, random_state=None)
mnist_train_split, mnist_val = valid_split(mnist_train)

BATCH_SIZE = 32
# TODO: create the three DataLoaders (which ones should shuffle?)
mnist_train_loader = ...
mnist_val_loader = ...
mnist_test_loader = ...

print(f"train: {len(mnist_train_split)} - val: {len(mnist_val)} - test: {len(mnist_test)}")

In [ ]:
# ✅ Check
x, y = next(iter(mnist_train_loader))
print("one mini-batch:", x.shape, y.shape)
assert x.shape == (BATCH_SIZE, 1, 28, 28) and y.shape == (BATCH_SIZE,), "wrong mini-batch shape: check batch_size"
assert len(mnist_train_loader.dataset) == 48000 and len(mnist_val_loader.dataset) == 12000 and len(mnist_test_loader.dataset) == 10000
print("All good!")

## MLPs

From the past lectures you should remember that, a torch model is a `class` inheriting from `torch.nn.Module`. Inside this class, we'll define two methods:
* the constructor (`__init__`) in which we define the building blocks of our model as class variables;
* the `forward` method, which specifies how the data fed into the model needs to be processed in order to produce the output.

Note for those who already know something about NNs: we don't need to define `backward` methods since we're constructing our model with built-in PT building blocks. PyTorch automatically creates a `backward` routine based upon the `forward` method.

`torch.nn.Linear(in_features, out_features, bias=True)` applies an affine (linear + shift) transformation to its input:
$$y = xW^T + b$$

- `in_features`: size of each input sample, i.e. the number of features of $x$.
- `out_features`: size of each output sample, i.e. the number of neurons in the layer.
- `bias`: if `False`, the layer does not learn $b$ (so $y = xW^T$).

The learnable parameters are:
- `weight` $W$, with shape `(out_features, in_features)`: row $j$ holds the weights of neuron $j$.
- `bias` $b$, with shape `(out_features,)`: one value per neuron.

Shapes: the input $x$ has shape `(..., in_features)` and the output $y$ has shape `(..., out_features)`. Usually `...` is the batch size, so every sample in the batch is transformed independently with the same $W$ and $b$.

**An MLP is the mathematical composition operation of $N$ `Linear` layers with a non linear activation between each hidden layer.**

<p align="center">
  <img src="notebooks_imgs/mlp_with_nonlinear_activations.png" alt="drawing" width="500" style="background-color: white;"/>
</p>

### 🛠 Exercise 5 — The MLP

Complete the `MLP` class following the figure above.

**`__init__`**: define
- three `torch.nn.Linear` layers: `layer1` (`in_features` → `hidden1`), `layer2` (`hidden1` → `hidden2`), `layer3` (`hidden2` → `out_features`);
- the activation `torch.nn.ReLU()`.

**`forward`**:
1. flatten each image `(1, 28, 28)` into a vector of 784 values: `x.reshape(x.shape[0], -1)` keeps the batch dimension and merges all the others (`-1` means "whatever size is needed");
2. apply `layer1` and then the activation; then `layer2` and the activation;
3. apply `layer3` **without** activation and return the result: these are the **logits**, one raw score per class. No softmax: the loss function applies it internally (see *Cross-entropy loss* below).

Finally create the model with `in_features=28*28`, `hidden1=64`, `hidden2=64`, `out_features=10` and move it to `DEVICE` with `.to(DEVICE)`.

The next cell checks the shape of the output: it must be `torch.Size([32, 10])`, one row of 10 scores per image.

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, in_features: int, hidden1: int, hidden2: int, out_features: int):
        super().__init__()
        # TODO: define the three linear layers (with bias) and the activation
        self.layer1 = ...  # input    -> hidden 1
        self.layer2 = ...  # hidden 1 -> hidden 2
        self.layer3 = ...  # hidden 2 -> output
        self.activation = ...

    def forward(self, x):
        # TODO: input layer (identity): flatten each image into a vector of 784 raw features
        x = ...                                  # (batch, 1, 28, 28) -> (batch, 784)
        # TODO: hidden layer 1: z = xW1^T + b1, a = ReLU(z)
        x = ...                                  # (batch, 784) -> (batch, hidden1)
        # TODO: hidden layer 2: z = xW2^T + b2, a = ReLU(z)
        x = ...                                  # (batch, hidden1) -> (batch, hidden2)
        # TODO: output layer: z = xW3^T + b3, raw scores (logits), one per class (no activation!)
        logits = ...                             # (batch, hidden2) -> (batch, n_classes)
        return logits


# TODO: create the model (784 -> 64 -> 64 -> 10) and move its parameters to DEVICE
model = ...

In [ ]:
# check the model output size
fake_input = torch.randn(BATCH_SIZE, 1, 28, 28, device=DEVICE)
model(fake_input).size()

We can create an instance of our model and inspect the current parameters by using the `state_dict` method, which prints the building blocks of our model and their current parameters. Note that `state_dict` is essentially a dictonary indexed by the names of the building blocks which we defined inside the constructor (plus some additional identifiers if a layer has more than one set of parameters).

In [ ]:
for param_name, param in model.state_dict().items():
    print(param_name, param)

# Training a Neural Network

## Automatic differentiation in torch


PyTorch is built with support for differentiation in mind.
In the end, Deep Learning (for now) is all about differentiation and building cascades of differentiable function into complicated multilayer deep neural networks.

Essentially, all PyTorch built-ins support differentiability (unless the function is not differentiable, of course).
Today we will see how to compute derivatives in PyTorch.


Under the hood, each torch `Tensor` has a boolean attribute `requires_grad`, which tells `autograd` whether it should keep trace of operations on the tensor or not. `autograd` is the automatic differentiation engine of PyTorch.

To store gradients of intermediate computations, we can call `.retain_grad()` on the intermediate node.

## Optimizers

Now that we have the tools to compute gradients of any function (specifically, we will be interested in loss functions), it is time to figure out what to do with these gradients.

In torch, it is straightforward to use most optimization techniques based on Gradient Descent and its variations, such as SGD, Adam, RMSProp etc.

To do so, you should exploit the tools of the `torch.optim` package, that contains the most famous training algorithms in the `Optimizer` class. To cite some:

*   `torch.optim.SGD`
*   `torch.optim.Adam`
*   `torch.optim.RMSprop`
*   `torch.optim.Adagrad`

To construct an Optimizer you have to give it an iterable containing the parameters to optimize. Then, you can specify optimizer-specific options such as the learning rate, weight decay, etc.

### SGD and its main parameters

`torch.optim.SGD(params, lr, momentum=0, ...)`

Plain (stochastic) gradient descent moves every parameter $\theta$ a small step against its gradient $g = \nabla_\theta L$, computed on a mini-batch:
$$\theta \leftarrow \theta - \text{lr} \cdot g$$

- `params`: the tensors to optimize, usually `model.parameters()`.
- `lr` (learning rate): the size of each step. Too large and the loss oscillates or diverges; too small and training is very slow. It is the most important hyperparameter to tune.
- `momentum` ($\mu$, typically `0.9`): instead of following only the current gradient, SGD keeps a running "velocity" $v$ that remembers past gradients:
$$v \leftarrow \mu \, v + g, \qquad \theta \leftarrow \theta - \text{lr} \cdot v$$
  Like a ball rolling downhill, it speeds up along directions where gradients consistently agree and damps the zig-zag where they keep changing sign. With `momentum=0` we get plain SGD.

### Why Adam is useful

SGD uses **the same learning rate for every parameter**. In a neural network, however, gradients can have very different scales across layers and parameters: some weights receive large gradients, others tiny or rare ones. A single `lr` is then too large for some and too small for others.

In practice this means:
- **fast convergence** without much tuning: the default `lr=1e-3` works well on many problems;
- **robustness** to badly scaled gradients and to sparse or noisy gradients;
- it combines the benefits of **momentum** (via $m$) and of **RMSprop** (via $s$).

### 🛠 Exercise 6 — The optimizer

Create an SGD optimizer (`torch.optim.SGD`) for the parameters of `model` (`model.parameters()`), with learning rate `lr=1e-3` and `momentum=0.9`.

In [ ]:
optimizer = ...  # TODO

Using an optimizer is very simple, there are three main steps:

*   delete the current gradient information on the parameters: `optimizer.zero_grad()`
*   compute the derivatives of the loss: `loss.backward()`
*   perform a gradient descent step and update parameters (according to the algorithm in use): `optimizer.step()`

Note that the first step should always be performed since, as we saw, torch accumulates gradients.


### Definition of a loss function

One key element that we need to train any neural network is a loss function, i.e. a function that quantifies how *good* is our fit to the data and that is differentiable w.r.t. the weights and biases of the network.

We saw some examples of common loss functions in the lecture, and all the main losses used in Deep Learning are already implemented and available in PyTorch, to cite some:

*   `torch.nn.MSELoss`
*   `torch.nn.CrossEntropyLoss`
*   `torch.nn.BCELoss`
*   `torch.nn.KLDivLoss`

You can also define your own custom loss function, and as long as you use built-in torch functions to compute it (and you keep it differentiable), you should be fine.

### Cross-entropy loss

This is a classification problem, so we use `torch.nn.CrossEntropyLoss`.

**From scores to probabilities.** For every image our model outputs $K = 10$ raw scores $z_1, \dots, z_K$, one per class, called **logits**. They can be any real number. The **softmax** function turns them into probabilities (positive numbers that sum to 1):
$$p_k = \text{softmax}(z)_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$
The larger the score $z_k$, the larger the probability $p_k$ that the image belongs to class $k$.

**The loss.** If the true class of the image is $y$, the cross-entropy is minus the log of the probability that the model gives to the **correct** class:
$$\mathcal{L} = -\log p_y$$
and on a mini-batch of $N$ images we take the mean:
$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log p_{i,\, y_i}$$

**Intuition.** The loss measures how *surprised* the model is by the correct answer:

| Probability given to the correct class $p_y$ | Loss $-\log p_y$ | Meaning |
|---|---|---|
| 0.9 | 0.11 | confident and right: small loss |
| 0.5 | 0.69 | unsure |
| 0.1 | 2.30 | same as a random guess among 10 classes |
| 0.01 | 4.61 | confident and **wrong**: large loss |

- The loss is 0 only when the model gives probability 1 to the correct class.
- It grows very quickly when the model is confidently wrong ($-\log p \to \infty$ when $p \to 0$), so these mistakes are punished the most.
- Minimizing it means pushing up the score of the correct class and pushing down the others.
- An untrained model guesses at random ($p_y \approx 1/10$), so **the training loss starts around $\ln 10 \approx 2.3$**. This is a useful sanity check.

**In PyTorch.** `torch.nn.CrossEntropyLoss` takes as input the **raw logits** (shape `(batch, K)`) and the **integer labels** (shape `(batch,)`, values in $0, \dots, K-1$). It applies the softmax internally, so our model must **not** end with a softmax.

### 🛠 Exercise 7 — The loss function

Create the cross-entropy loss, an instance of `torch.nn.CrossEntropyLoss`, and call it `loss`. It is an object that we use as a function: `loss(logits, labels)` returns the mean loss over the mini-batch.

In [ ]:
loss = ...  # TODO

### 🛠 Exercise 8 — Cross-entropy by hand

Let's verify that `CrossEntropyLoss` computes exactly the formula above. With one image, 3 classes and true class 0:
1. turn the logits into probabilities with `torch.softmax`, along the dimension of the classes (`dim=1`; `dim=0` is the batch);
2. compute $-\log p_y$ by hand: take the probability of the correct class, `probs[0, target[0]]`, and apply `-torch.log(...)`;
3. compare it with the value returned by `loss(logits, target)`, computed from the raw logits: the two values must be the same.

In [ ]:
# a small example: one image, 3 classes, the true class is 0
logits = torch.tensor([[2.0, 0.5, -1.0]])  # raw scores of the model, shape (batch=1, K=3)
target = torch.tensor([0])                  # true class, shape (batch=1,)

probs = ...     # TODO: softmax over the classes
by_hand = ...   # TODO: -log of the probability of the correct class

print("probabilities:", probs)                                  # p_0 should be about 0.79
print("by hand -log(p_y):", by_hand.item())
print("CrossEntropyLoss: ", loss(logits, target).item())        # same value, computed from the logits

## Training code

The following cell contains the nested loop that we will run to train the network. Its pseudocode is as follows:


```
Loop over epochs:
    Loop over data:
        Perform a forward pass
        Compute the loss
        Erase the past gradients
        Compute gradients performing a backward pass
        Update the parameters
```



SGD training is an iterative process, that we repeat for a (usually) fixed number of *epochs*. In each epoch we traverse the whole dataset by exploiting our `DataLoader` object, that provides us with randomly drawn mini-batches of 8 elements.

We can keep track of the loss so that we can compare it with the loss of the least squares estimate we obtained before.

### 🛠 Exercise 9 — Training and evaluation functions

Complete the two functions that we will reuse for **every** model of this lab.

**`train_one_epoch`**: for every mini-batch `(x, y)` of the dataloader, translate the pseudocode above into code:
1. move `x` and `y` to `DEVICE` (the model is already there);
2. forward pass: compute the output of the model on `x`;
3. compute the loss `l` between the output and `y` with `loss_fn`;
4. erase the past gradients, compute the new ones with a backward pass, update the parameters: these are the three optimizer steps of the *Optimizers* section, and **the order matters**;
5. `epoch_loss += l.item()` is given: `.item()` turns a 1-element tensor into a Python number.

**`evaluate`**: the same forward pass, but without gradients and without updates. For every mini-batch:
- accumulate the loss: `loss_fn` returns the **mean** over the batch, so multiply it by the batch size `len(y)` to get the sum (the last batch can be smaller than the others!);
- count the correct predictions: the predicted class is the index of the largest logit, `out.argmax(dim=1)`. Compare it with `y` (`==`), sum the `True`s with `.sum()` and convert to a number with `.item()`.

`model.train()`, `model.eval()` and `torch.no_grad()` are already there: make sure you understand why we need them.

In [ ]:
def train_one_epoch(model, dataloader, loss_fn, optimizer):
    """Train `model` for one epoch (one full pass over `dataloader`)
    and return the average loss over the mini-batches."""
    model.train()  # set the model in training mode
    epoch_loss = 0  # we accumulate the loss on all the mini-batches to average it
    for x, y in dataloader:
        x, y = ...               # TODO: move the mini-batch to the same device as the model
        out = ...                # TODO: forward pass
        l = ...                  # TODO: compute the loss
        ...                      # TODO: erase the past gradients
        ...                      # TODO: backward pass: compute the gradients
        ...                      # TODO: update the parameters
        epoch_loss += l.item()
    return epoch_loss / len(dataloader)  # len(dataloader) is the number of mini-batches


def evaluate(model, dataloader, loss_fn):
    """Evaluate `model` on `dataloader` without updating it
    and return the average loss and the accuracy."""
    model.eval()  # set the model in evaluation mode
    total_loss, correct = 0, 0
    with torch.no_grad():  # no gradients needed: lighter and no accidental updates
        for x, y in dataloader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            total_loss += ...    # TODO: loss of the batch (a Python number) times the batch size
            correct += ...       # TODO: number of correct predictions in the batch (a Python number)
    n_samples = len(dataloader.dataset)
    return total_loss / n_samples, correct / n_samples

In [ ]:
# ✅ Check: the model is not trained yet, so it should guess at random
val_loss, val_acc = evaluate(model, mnist_val_loader, loss)
print(f"untrained model - val loss: {val_loss:.3f} (expected about ln(10) = 2.30) - val acc: {val_acc:.3f} (expected about 0.10)")
assert 2.0 < val_loss < 2.6 and val_acc < 0.3, "unexpected values: check evaluate"
print("All good!")

### 🛠 Exercise 10 — The training loop

Train `model` for 5 epochs. At every epoch:
1. train for one epoch on `mnist_train_loader` with `train_one_epoch` (pass `model`, the dataloader, `loss` and `optimizer`);
2. evaluate on `mnist_val_loader` with `evaluate`;
3. append the three values to `train_losses`, `val_losses` and `val_accs`: we need them for the plots.

After the loop, evaluate the model **once** on the test set (`mnist_test_loader`).

Expected: the training loss decreases at every epoch, and after 5 epochs the validation and test accuracy are around 95%.

In [ ]:
epochs = 5
train_losses, val_losses, val_accs = [], [], []
for epoch in range(epochs):
    train_loss = ...            # TODO: train for one epoch
    val_loss, val_acc = ...     # TODO: evaluate on the validation set
    # TODO: append the three values to their lists
    ...
    print(f"Epoch {epoch + 1}/{epochs} - train loss: {train_loss:.4f} - val loss: {val_loss:.4f} - val acc: {val_acc:.4f}")

# TODO: after training, evaluate once on the test set
test_loss, test_acc = ...
print(f"Test loss: {test_loss:.4f} - test accuracy: {test_acc:.4f}")

### Plot Training curves

In [ ]:
def plot_curves(curves, title, ylabel, reference=None):
    """Plot one or more lists of numbers (one value per epoch).

    curves:    dict {label: list of values}, e.g. {"train": train_losses, "validation": val_losses}
    reference: optional (label, value) drawn as a horizontal dashed line, e.g. ("test", test_loss)
    """
    fig, ax = plt.subplots(figsize=(6, 4))
    for label, values in curves.items():
        epochs_axis = range(1, len(values) + 1)  # epochs start from 1
        ax.plot(epochs_axis, values, marker="o", label=label)
    if reference is not None:
        ref_label, ref_value = reference
        ax.axhline(ref_value, color="gray", linestyle="--", label=ref_label)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))  # only integer epochs on the x axis
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()


plot_curves({"train": train_losses, "validation": val_losses},
            title="MNIST MLP - loss", ylabel="Cross-entropy loss",
            reference=("test", test_loss))

plot_curves({"validation": val_accs},
            title="MNIST MLP - validation accuracy", ylabel="Accuracy",
            reference=("test", test_acc))

### Plot confusion matrix

The confusion matrix shows, for every true class (rows), how the model classified its images (columns).

- The **diagonal** contains the correctly classified images.
- The **off-diagonal** cells are the errors: the cell in row $i$, column $j$ counts the images of digit $i$ predicted as digit $j$.

With `normalize="true"` each row sums to 1, so the diagonal is the accuracy on each digit. This makes it easy to see which digits are the hardest and which pairs are confused most often (e.g. 4 and 9, 3 and 5).

### 🛠 Exercise 11 — Predictions for the confusion matrix

Complete `get_predictions`: for every mini-batch, append to `y_pred` the predicted classes, i.e. the index of the largest logit (as in `evaluate`).

scikit-learn works with NumPy, i.e. on the CPU: move the predictions back with `.cpu()` before appending them. (The true labels `y` were never moved to the GPU, so they are fine as they are.)

`plot_confusion_matrix` is given.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


def get_predictions(model, dataloader):
    """Return the true labels and the predicted labels of `model` on `dataloader`."""
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in dataloader:
            out = model(x.to(DEVICE))
            y_true.append(y)
            y_pred.append(...)  # TODO: predicted class = index of the largest logit, back on the CPU
    return torch.cat(y_true), torch.cat(y_pred)


def plot_confusion_matrix(y_true, y_pred, title, normalize=None):
    """Plot the confusion matrix: rows are the true classes, columns the predicted ones.

    normalize: None -> raw counts, "true" -> each row sums to 1 (fraction of each true class)
    """
    cm = confusion_matrix(y_true, y_pred, normalize=normalize)
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(cm, display_labels=list(range(10))).plot(
        ax=ax, cmap="Blues", values_format=".2f" if normalize else "d", colorbar=True
    )
    ax.set_title(title)
    plt.show()


y_true, y_pred = get_predictions(model, mnist_test_loader)
plot_confusion_matrix(y_true, y_pred, title="Test confusion matrix (counts)")
plot_confusion_matrix(y_true, y_pred, title="Test confusion matrix (fraction of each true class)", normalize="true")

## Hyperparameter tuning

What we would like to do is to find which is the best hidden sizes for the models selecting from (32, 64, 128).

To do it we use `sklearn.model_selection.GridSearchCV`:
- it trains a model for **every combination** of the hyperparameters in `param_grid` (here $3 \times 3 = 9$ combinations);
- each combination is evaluated with **k-fold cross-validation** (`cv=3`): the training set is split into 3 parts, the model is trained on 2 and validated on the remaining one, **rotating 3 times**. The score is the average validation accuracy;
- at the end (`refit=True`) the best combination is retrained on the whole training set, and we evaluate it **once** on the test set.

`GridSearchCV` only works with scikit-learn estimators, so we wrap our PyTorch `MLP` with `skorch.NeuralNetClassifier`, which adds the `fit`/`predict` methods and runs the training loop for us.

**Note:** this takes a while, since it trains $9 \times 3 + 1 = 28$ models for 5 epochs each.

### 🛠 Exercise 12 — Grid search

The data preparation and the `NeuralNetClassifier` wrapper are given: read the comments, especially about the `module__` and `optimizer__` prefixes. Complete:

1. `param_grid`: a dictionary where every **key** is the name of a parameter of `net` (with its prefix) and every **value** is the list of values to try. We want `module__hidden1` and `module__hidden2`, both in `[32, 64, 128]`.
2. `grid`: a `GridSearchCV` built from `net` and `param_grid`, with 3-fold cross-validation (`cv=3`), `scoring="accuracy"`, `refit=True` (retrain the best combination on all the data at the end) and `verbose=2` (to follow the progress).
3. Train it with `grid.fit(X_train, y_train)`: exactly like any scikit-learn model.

The code that prints the results is given.

In [ ]:
from sklearn.model_selection import GridSearchCV
from skorch import NeuralNetClassifier

# GridSearchCV works with arrays, so we build X (images) and y (labels) from the whole training set
# applying the same preprocessing as our transforms: scale pixels to [0, 1], then normalize with mean/std.
# The test set is NOT used here: it is kept for the final evaluation of the best model.
X_train = ((mnist_train.data.float() / 255 - mean) / std).unsqueeze(1).numpy()  # (60000, 1, 28, 28)
y_train = mnist_train.targets.numpy()                                          # (60000,)

# skorch wraps our PyTorch model into a scikit-learn estimator (with .fit / .predict),
# so that it can be used inside GridSearchCV.
# Parameters with the "module__" prefix are passed to MLP.__init__,
# parameters with the "optimizer__" prefix are passed to the optimizer.
net = NeuralNetClassifier(
    module=MLP,
    module__in_features=28*28,
    module__hidden1=64,                    # overwritten by the grid
    module__hidden2=64,                    # overwritten by the grid
    module__out_features=10,
    criterion=torch.nn.CrossEntropyLoss,   # the model outputs logits
    optimizer=torch.optim.SGD,
    optimizer__lr=1e-3,
    optimizer__momentum=0.9,
    max_epochs=2,
    batch_size=BATCH_SIZE,
    iterator_train__shuffle=True,          # shuffle the training mini-batches
    train_split=None,                      # no internal validation split: GridSearchCV does the cross-validation
    device=str(DEVICE),                    # skorch moves the model and the data to this device
    verbose=0,
)

# TODO: all the combinations of hidden sizes: 3 x 3 = 9 models
param_grid = {
    ...
}

# TODO: grid search with 3-fold cross-validation, accuracy as score, refit of the best model, verbose=2
# total: 9 combinations x 3 folds = 27 trainings (+ 1 final refit of the best model on all the data)
grid = ...
# TODO: run the grid search on X_train, y_train
...

print("Best hyperparameters:", grid.best_params_)
print(f"Best cross-validation accuracy: {grid.best_score_:.4f}")

# results of every combination, from the best to the worst
results = sorted(zip(grid.cv_results_["mean_test_score"], grid.cv_results_["std_test_score"], grid.cv_results_["params"]),
                 key=lambda r: r[0], reverse=True)
for mean_acc, std_acc, params in results:
    print(f"hidden1={params['module__hidden1']:>3} hidden2={params['module__hidden2']:>3} -> accuracy {mean_acc:.4f} ± {std_acc:.4f}")

### Retrain with best parameters

Now that we know the best hidden sizes, we build a new `MLP` with them and train it with our own training loop on the **full training set** (training + validation, 60k images): the validation set was only needed to choose the hyperparameters, so now we can use all the data to train.

The test set is used only once, at the end, to measure the final performance.

### 🛠 Exercise 13 — Retrain with the best hyperparameters

1. Read the best hidden sizes from `grid.best_params_`: a dictionary with the same keys as `param_grid`.
2. Create a new `MLP` with these sizes (on `DEVICE`) and a new SGD optimizer for **its** parameters, with the same `lr` and `momentum` as before.
3. Train it for 5 epochs on `mnist_train_loader` with `train_one_epoch`, and collect the training loss of every epoch in `final_train_losses`.

The final evaluation on the test set and the plots are given.

In [ ]:
# TODO: best hidden sizes found by the grid search
best_hidden1 = ...
best_hidden2 = ...
print(f"Retraining with hidden1={best_hidden1}, hidden2={best_hidden2}")

# TODO: new model and optimizer, same training setup as before
final_model = ...
final_optimizer = ...

epochs = 5
final_train_losses = []
for epoch in range(epochs):
    train_loss = ...  # TODO: train final_model for one epoch
    final_train_losses.append(train_loss)
    print(f"Epoch {epoch + 1}/{epochs} - train loss: {train_loss:.4f}")

# final evaluation, once, on the test set
final_test_loss, final_test_acc = evaluate(final_model, mnist_test_loader, loss)
print(f"Final model - test loss: {final_test_loss:.4f} - test accuracy: {final_test_acc:.4f}")

plot_curves({"train (full training set)": final_train_losses},
            title=f"Final MLP ({best_hidden1}, {best_hidden2}) - loss", ylabel="Cross-entropy loss",
            reference=("test", final_test_loss))

y_true, y_pred = get_predictions(final_model, mnist_test_loader)
plot_confusion_matrix(y_true, y_pred, title="Final model - test confusion matrix (fraction of each true class)", normalize="true")

# CNNs

## Why not an MLP for images?

Our MLP works, but it has two problems with images:

1. **It ignores the spatial structure.** The first thing it does is flatten the $28 \times 28$ image into a vector of 784 numbers: for the MLP, two neighbouring pixels are as unrelated as two pixels in opposite corners.
2. **It has many parameters.** Every neuron of the first layer is connected to all 784 pixels: with 128 neurons this is already $784 \times 128 \approx 100\text{k}$ weights, just for the first layer.

A **Convolutional Neural Network (CNN)** solves both problems with two ideas:

- **Local connections:** each neuron looks only at a small patch of the image (e.g. $3 \times 3$ pixels), because the important patterns (edges, strokes, corners) are local.
- **Weight sharing:** the *same* small set of weights (a **filter** or **kernel**) slides over the whole image. A filter that detects a vertical stroke is useful everywhere in the image, so there is no need to learn it again for every position.

## The building blocks

**Convolution** — `torch.nn.Conv2d(in_channels, out_channels, kernel_size, padding)`

A filter of size $k \times k$ slides over the image; at each position it computes a weighted sum of the pixels below it (plus a bias). The result is a new image, called **feature map**, that is bright where the pattern of the filter is present.

- `in_channels`: number of channels of the input (1 for grayscale MNIST, 3 for RGB images).
- `out_channels`: number of filters, i.e. how many different patterns the layer learns. Each filter produces one feature map.
- `kernel_size`: size $k$ of the filter (e.g. 3 means $3 \times 3$).
- `padding`: pixels of zeros added around the border. With `kernel_size=3` and `padding=1` the output keeps the same height and width as the input.

Shapes: input `(batch, in_channels, H, W)` → output `(batch, out_channels, H, W)` (with `padding = kernel_size // 2`).

Number of parameters: `out_channels × (in_channels × k × k + 1)`. It does **not** depend on the size of the image!

**Max pooling** — `torch.nn.MaxPool2d(kernel_size=2)`

Splits each feature map into $2 \times 2$ blocks and keeps only the largest value of each block: it halves height and width, keeps the strongest responses and has no parameters. Shapes: `(batch, C, H, W)` → `(batch, C, H/2, W/2)`.

**Putting it together:** the first convolutional layers detect simple patterns (edges, strokes), the next ones combine them into more complex shapes (loops, corners, parts of digits). At the end, the feature maps are flattened and a `Linear` layer computes the scores of the 10 classes, exactly as in the MLP.

Our small CNN has two convolutional blocks (convolution → ReLU → max pooling) followed by a linear classifier:

| Layer | Output shape | Parameters |
|---|---|---|
| Input | `(batch, 1, 28, 28)` | – |
| `Conv2d(1, 16, kernel_size=3, padding=1)` + ReLU | `(batch, 16, 28, 28)` | $16 \times (1 \cdot 3 \cdot 3 + 1) = 160$ |
| `MaxPool2d(2)` | `(batch, 16, 14, 14)` | 0 |
| `Conv2d(16, 32, kernel_size=3, padding=1)` + ReLU | `(batch, 32, 14, 14)` | $32 \times (16 \cdot 3 \cdot 3 + 1) = 4640$ |
| `MaxPool2d(2)` | `(batch, 32, 7, 7)` | 0 |
| Flatten | `(batch, 1568)` | 0 |
| `Linear(1568, 10)` | `(batch, 10)` | $1568 \times 10 + 10 = 15690$ |

In total about **20k parameters**, around 5 times fewer than our best MLP.

### 🛠 Exercise 14 — The CNN

Complete the `CNN` class following the table above.

**`__init__`**:
- `conv1`: a `torch.nn.Conv2d` from 1 channel (grayscale) to `n_filters1` channels, with `kernel_size=3` and `padding=1` (so height and width do not change);
- `conv2`: the same, from `n_filters1` to `n_filters2` channels;
- `pool`: a `torch.nn.MaxPool2d` with `kernel_size=2` (halves height and width). One pooling layer is enough: it has no parameters, so we can reuse it twice;
- `activation`: a ReLU;
- `classifier`: a `torch.nn.Linear` from the flattened feature maps to `out_features`. How many numbers are left after the second pooling? `n_filters2` feature maps of size $7 \times 7$.

**`forward`**: twice *convolution → ReLU → pooling*, then flatten (as in the MLP, but now **after** the convolutions), then the classifier.

Use the shapes in the comments (and in the table) to check every step. The next cell prints the output size and compares the number of parameters with the best MLP.

In [ ]:
class CNN(torch.nn.Module):
    def __init__(self, n_filters1: int = 16, n_filters2: int = 32, out_features: int = 10):
        super().__init__()
        # TODO: convolutional block 1: (batch, 1, 28, 28) -> (batch, n_filters1, 14, 14)
        self.conv1 = ...
        # TODO: convolutional block 2: (batch, n_filters1, 14, 14) -> (batch, n_filters2, 7, 7)
        self.conv2 = ...
        self.pool = ...        # TODO: halves height and width
        self.activation = ...  # TODO
        # TODO: classifier: (batch, n_filters2 * 7 * 7) -> (batch, out_features)
        self.classifier = ...

    def forward(self, x):
        # here we do NOT flatten the input: the convolutions work directly on the 2D image
        x = ...       # TODO: conv1 -> ReLU -> pool: (batch, 1, 28, 28)  -> (batch, 16, 14, 14)
        x = ...       # TODO: conv2 -> ReLU -> pool: (batch, 16, 14, 14) -> (batch, 32, 7, 7)
        x = ...       # TODO: flatten: (batch, 32, 7, 7) -> (batch, 1568)
        logits = ...  # TODO: (batch, 1568) -> (batch, 10)
        return logits


cnn_model = CNN().to(DEVICE)
print(cnn_model)

### 🛠 Exercise 15 — Counting the parameters

Complete `count_parameters`: `model.parameters()` returns all the parameter tensors of a model (weights and biases of every layer), and `p.numel()` returns the number of elements of the tensor `p`. Sum them all.

Expected: `20,490` for the CNN, the total of the table above ($160 + 4640 + 15690$).

In [ ]:
# check the model output size
fake_input = torch.randn(BATCH_SIZE, 1, 28, 28, device=DEVICE)
print("output size:", cnn_model(fake_input).size())

# compare the number of parameters with the best MLP
def count_parameters(model):
    return ...  # TODO: total number of parameters of the model

print(f"CNN parameters: {count_parameters(cnn_model):,}")
print(f"MLP parameters: {count_parameters(final_model):,}")

### Training the CNN

Since `train_one_epoch`, `evaluate`, `plot_curves` and `plot_confusion_matrix` work with **any** model, we can reuse them as they are: we only need a new optimizer for the parameters of the CNN.

We use the same setup as the MLP (SGD with `lr=1e-3` and `momentum=0.9`, 5 epochs, same training/validation split), so that the comparison between the two models is fair.

### 🛠 Exercise 16 — Training the CNN

Same as Exercise 10, with the CNN:
1. create `cnn_optimizer`, an SGD optimizer on the parameters of `cnn_model`, with `lr=1e-3` and `momentum=0.9`;
2. in every epoch, train on `mnist_train_loader`, evaluate on `mnist_val_loader` and append the results to `cnn_train_losses`, `cnn_val_losses` and `cnn_val_accs`. Use the names `train_loss`, `val_loss`, `val_acc` (they are used by the `print`);
3. after the loop, evaluate the CNN once on the test set.

Expected: about 98% test accuracy after 5 epochs.

In [ ]:
cnn_optimizer = ...  # TODO

epochs = 5
cnn_train_losses, cnn_val_losses, cnn_val_accs = [], [], []
for epoch in range(epochs):
    # TODO: train for one epoch, evaluate on the validation set and store the three values
    ...
    print(f"Epoch {epoch + 1}/{epochs} - train loss: {train_loss:.4f} - val loss: {val_loss:.4f} - val acc: {val_acc:.4f}")

# TODO: after training, evaluate once on the test set
cnn_test_loss, cnn_test_acc = ...
print(f"CNN - test loss: {cnn_test_loss:.4f} - test accuracy: {cnn_test_acc:.4f}")

### CNN training curves and confusion matrix

In [ ]:
plot_curves({"train": cnn_train_losses, "validation": cnn_val_losses},
            title="MNIST CNN - loss", ylabel="Cross-entropy loss",
            reference=("test", cnn_test_loss))

plot_curves({"validation": cnn_val_accs},
            title="MNIST CNN - validation accuracy", ylabel="Accuracy",
            reference=("test", cnn_test_acc))

y_true, y_pred = get_predictions(cnn_model, mnist_test_loader)
plot_confusion_matrix(y_true, y_pred, title="CNN - test confusion matrix (fraction of each true class)", normalize="true")

### MLP vs CNN

Compare the test accuracy of the CNN with the one of the best MLP: what do you notice, considering the number of parameters of the two models? Which digits are still confused by the CNN?

In [ ]:
print(f"Best MLP - parameters: {count_parameters(final_model):>7,} - test accuracy: {final_test_acc:.4f}")
print(f"CNN      - parameters: {count_parameters(cnn_model):>7,} - test accuracy: {cnn_test_acc:.4f}")

# Homework

## 1. MLP on FashionMNIST

[FashionMNIST](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.FashionMNIST.html) is a drop-in replacement for MNIST: same format ($28 \times 28$ grayscale images, 60k training + 10k test, 10 classes), but the images are clothes instead of digits, and the task is harder.

| Label | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|---|
| Class | T-shirt/top | Trouser | Pullover | Dress | Coat | Sandal | Shirt | Sneaker | Bag | Ankle boot |

Repeat the MNIST pipeline of this notebook on FashionMNIST:
1. Load the dataset with `torchvision.datasets.FashionMNIST` and compute **its own** normalization values with `find_normalization_values` (do not reuse the MNIST ones).
2. Split training and validation sets with `ValidSplit` and create the `DataLoader`s.
3. Train the `MLP` with `train_one_epoch` / `evaluate`, plot the training curves with `plot_curves` and evaluate once on the test set.
4. Run `GridSearchCV` on the hidden sizes, retrain the best model on the full training set and compare it with the MLP you trained in step 3.
5. Plot the confusion matrix with the class names as labels.

Questions:
- How does the test accuracy compare with MNIST? Why?
- Which classes are confused the most? Does it make sense, looking at some images of those classes?
- Is the best hidden size the same as for MNIST?

## 2. (Advanced) Cross-validation with CNNs

Use `GridSearchCV` + `skorch.NeuralNetClassifier` to tune the `CNN` of this notebook on FashionMNIST:
1. Wrap the `CNN` in a `NeuralNetClassifier` (remember `device=str(DEVICE)`).
2. Search over the number of filters, e.g. `module__n_filters1` in `[8, 16, 32]` and `module__n_filters2` in `[16, 32, 64]`, and optionally the learning rate `optimizer__lr` in `[1e-3, 1e-2]`.
3. Use `cv=3` and report the mean ± std validation accuracy of every combination.
4. Retrain the best CNN on the full training set and evaluate it once on the test set.

Questions:
- Does increasing the number of filters always help? Compare accuracy and number of parameters (`count_parameters`).
- How does the best CNN compare with the best MLP?

**Tip:** a grid search trains *(number of combinations) × (number of folds)* models, so check the cost before running it: use a GPU and start with few epochs or a subset of the training set.